# 02 Pair Selection

Generate correlation candidates and apply fixed-orientation Engle–Granger tests. Holm correction covers the full formation asset-pair family. All decisions are saved for inspection.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_session import NotebookSession

session = NotebookSession.active()
cfg = session.config
RUN_DIR = session.run
session.begin('02_pairs', ['01_stock', '01_rf'])


## 2. Correlation candidates

Use training returns only. Each unordered candidate has one predetermined alphabetical regression direction.


In [ ]:
from src.Pair_Selection import compute_returns, correlation_matrix, generate_candidate_pairs
train_prices = session.frame('train_prices')
returns = compute_returns(train_prices)
correlations = correlation_matrix(returns)
candidate_pairs = generate_candidate_pairs(correlations, cfg.correlation_neighbors)
print(f'{len(candidate_pairs):,} candidate pairs from {len(train_prices.columns)} assets')
display(pd.DataFrame(candidate_pairs, columns=['dependent', 'independent']).head(10))


## 3. Cointegration and integration screens

This replaces ordinary residual ADF p-values and minimum-p orientation selection. ADF level/difference screens are diagnostics for the I(1) assumption; they do not prove it.


In [ ]:
from src.Cointegration import screen_cointegration
selected, spreads, audit = screen_cointegration(train_prices, candidate_pairs,
    significance=cfg.cointegration_alpha, integration_alpha=cfg.integration_alpha)
session.save('cointegration_audit', audit)
session.save('cointegrated_pairs', selected)
display(audit[['pair', 'pvalue', 'adjusted_pvalue', 'integration_screen', 'selected']].head(20))
print(f'{len(selected)} pairs retained')
if selected.empty:
    raise ValueError('No pairs passed. The audit is saved. Review the result before changing thresholds.')


## 4. Save fitted spreads

Each column is a selected pair’s formation log-price residual, with alpha and beta recorded in cointegrated_pairs.


In [ ]:
formation_spreads = pd.DataFrame({f'{dep}-{ind}': values for (dep, ind), values in spreads.items()})
session.save('formation_spreads', formation_spreads)
display(selected[['pair', 'alpha', 'beta', 'adjusted_pvalue']].head(10))
formation_spreads.iloc[:, :3].plot(figsize=(10, 4), title='Selected formation spreads')
plt.show()


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
session.finish('02_pairs')
